# Guide intelligent du Musée · 03 Diagnostic (jalon `m2`)

Suivre l'entraînement du réseau dense de `02_baseline`, diagnostiquer ses défauts à partir des courbes, les corriger un par un, puis analyser ses erreurs.

Contexte du projet, données et découpage : voir `01_donnees.ipynb`.

## 1. Mise en place

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader

from commun import (SEED, ArtworkDataset, DenseNet, fit, get_device, load_images, load_metadata,
                    normalization_stats, set_seed, split_indices)

set_seed()
DEVICE = get_device()
meta, CLASSES = load_metadata()
X_all, y_all = load_images(meta)
splits = split_indices(meta)
MEAN, STD = normalization_stats(X_all, splits["train"])

print(f"PyTorch {torch.__version__}, calculs sur : {DEVICE}")
print(f"{len(meta)} œuvres, {len(CLASSES)} styles ; " + ", ".join(f"{k} : {len(i)}" for k, i in splits.items()))

## 2. Courbes de suivi et diagnostic

Le réseau de référence de `02_baseline` (section 3.4) est réentraîné à l'identique pour suivre, à chaque époque, la perte et l'accuracy en train et en validation : pixels aplatis, une couche cachée de 128 neurones en sigmoïde, SGD avec un taux d'apprentissage de 0,1, batch de 64, 30 époques, graine 0, sur CPU. Le test n'est pas utilisé.

In [ ]:
N_IN = 3 * 128 * 128
HIDDEN = [128]
LR = 0.1
BATCH_SIZE = 64
N_EPOCHS = 30
CPU = torch.device("cpu")
LN_C = math.log(len(CLASSES))  # perte d'un réseau qui répond au hasard : ln 5

datasets = {k: ArtworkDataset(X_all, y_all, splits[k], MEAN, STD) for k in ("train", "val")}
val_loader = DataLoader(datasets["val"], batch_size=256)
loss_fn = nn.CrossEntropyLoss()


def train_dense(hidden: list[int], lr: float, seed: int = SEED) -> tuple[DenseNet, dict[str, list[float]]]:
    """Entraîne un réseau dense par SGD et renvoie le modèle et son historique par époque."""
    set_seed(seed)
    model = DenseNet(N_IN, hidden, len(CLASSES))
    train_loader = DataLoader(datasets["train"], batch_size=BATCH_SIZE, shuffle=True)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    return model, fit(model, train_loader, val_loader, loss_fn, optimizer, N_EPOCHS, CPU)


def summarize(history: dict[str, list[float]]) -> dict[str, float]:
    """Chiffres du diagnostic : valeurs à la dernière époque et minimum de la perte de validation."""
    best = int(np.argmin(history["val_loss"]))
    return {"perte_train": history["train_loss"][-1], "perte_val": history["val_loss"][-1],
            "acc_train": history["train_acc"][-1], "acc_val": history["val_acc"][-1],
            "epoque_min_perte_val": best + 1, "min_perte_val": history["val_loss"][best]}

### 2.1 Réseau de référence

In [ ]:
baseline, history = train_dense(HIDDEN, LR)
ref = summarize(history)

print(f"Dernière époque : perte {ref['perte_train']:.3f} en train, {ref['perte_val']:.3f} en validation "
      f"(écart {ref['perte_val'] - ref['perte_train']:.3f}) ; accuracy {ref['acc_train']:.3f} en train, "
      f"{ref['acc_val']:.3f} en validation (écart {ref['acc_train'] - ref['acc_val']:.3f})")
print(f"Minimum de la perte de validation : {ref['min_perte_val']:.3f} à l'époque {ref['epoque_min_perte_val']} ; "
      f"perte d'un réseau qui répond au hasard : ln 5 = {LN_C:.3f}")
print(f"Meilleure accuracy de validation : {max(history['val_acc']):.3f} à l'époque {int(np.argmax(history['val_acc'])) + 1}")

In [ ]:
# Palette de référence (créneaux 1 et 2), encre et grille discrètes
COLORS = {"train": "#2a78d6", "val": "#eb6834"}
LABELS = {"train": "train", "val": "validation"}
INK, MUTED = "#52514e", "#898781"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.titlesize": 11, "font.size": 10,
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 1.5,
})


def plot_curves(ax, history: dict[str, list[float]], metric: str, chance: float) -> None:
    """Courbes train et validation d'une métrique ('loss' ou 'acc'), étiquetées en bout de ligne,
    avec en pointillés la valeur d'un réseau qui répond au hasard."""
    epochs = np.arange(1, len(history[f"train_{metric}"]) + 1)
    ends = {split: history[f"{split}_{metric}"][-1] for split in ("train", "val")}
    low, high = sorted(ends, key=ends.get)
    for split in ("train", "val"):
        ax.plot(epochs, history[f"{split}_{metric}"], color=COLORS[split], label=LABELS[split])
        nudge = -6 if split == low else 6  # écarte les deux étiquettes quand les courbes finissent proches
        ax.annotate(LABELS[split], (epochs[-1], ends[split]), xytext=(5, nudge), textcoords="offset points",
                    va="center", color=INK, fontsize=9)
    ax.axhline(chance, color=MUTED, linestyle="--", linewidth=1, label="réponse au hasard")
    ax.set_xlabel("époque")
    ax.set_xlim(1, epochs[-1] * 1.18)  # place pour les étiquettes en bout de ligne
    ax.set_xticks([1, *range(5, epochs[-1] + 1, 5)])


fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(11, 3.8))
plot_curves(ax_loss, history, "loss", LN_C)
e = ref["epoque_min_perte_val"]
ax_loss.plot(e, ref["min_perte_val"], "o", color=COLORS["val"], markersize=6)
ax_loss.annotate(f"minimum, époque {e}", (e, ref["min_perte_val"]), xytext=(8, -20), textcoords="offset points",
                 color=INK, fontsize=9)
ax_loss.set_ylabel("perte (entropie croisée)")
ax_loss.set_title("Perte")
plot_curves(ax_acc, history, "acc", 1 / len(CLASSES))
ax_acc.set_ylabel("accuracy")
ax_acc.set_title("Accuracy")
fig.legend(*ax_loss.get_legend_handles_labels(), loc="upper right", ncol=3, frameon=False)
fig.suptitle("Réseau de référence, par époque", x=0.01, ha="left", fontsize=12)
fig.tight_layout()
plt.show()

**Diagnostic du réseau de référence : sur-apprentissage, dès les premières époques.**

- La perte de train descend régulièrement jusqu'à 0,499, mais la perte de validation atteint son minimum dès l'époque 4 (1,641), puis remonte en oscillant pour finir à 1,955. À la dernière époque, l'écart entre train et validation atteint 1,457 en perte et 0,503 en accuracy (0,850 contre 0,347) : le réseau apprend le train par cœur au lieu de généraliser.
- La perte de validation finit au-dessus de ln 5 (1,955 contre 1,609) : le réseau fait pire, en perte, qu'une réponse uniforme. Ses erreurs sont commises avec assurance, et l'entropie croisée les punit fortement, alors que son accuracy reste au-dessus du hasard (0,347 contre 0,20).
- Perte et accuracy de validation ne désignent pas la même époque : minimum de perte à l'époque 4, meilleure accuracy (0,360) à l'époque 18. Avec 300 œuvres, les deux courbes de validation sont très bruitées d'une époque à l'autre.
- Le taux d'apprentissage n'est pas trop fort : la perte de train descend sans sauts, contrairement au premier échec provoqué ci-dessous.

### 2.2 Échecs provoqués

Trois runs qui ne changent qu'une chose par rapport à la référence, pour reconnaître sur les courbes les pannes classiques d'un entraînement :

- **taux d'apprentissage trop fort** : 3 au lieu de 0,1 ;
- **taux d'apprentissage trop faible** : 0,001 au lieu de 0,1 ;
- **réseau profond en sigmoïde** : 6 couches cachées de 128 neurones au lieu d'une.

In [ ]:
FAILURES = {
    "lr trop fort (3)": (HIDDEN, 3.0),
    "lr trop faible (0,001)": (HIDDEN, 0.001),
    "6 couches en sigmoïde": ([128] * 6, LR),
}
failure_histories = {name: train_dense(hidden, lr)[1] for name, (hidden, lr) in FAILURES.items()}

table = pd.DataFrame({"référence (lr 0,1)": ref} | {name: summarize(h) for name, h in failure_histories.items()}).T
table.round(3)

In [ ]:
fig, axes = plt.subplots(1, len(failure_histories), figsize=(13, 3.6))
for ax, (name, h) in zip(axes, failure_histories.items()):
    plot_curves(ax, h, "loss", LN_C)
    ax.set_title(name)
axes[0].set_ylabel("perte (entropie croisée)")
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper right", ncol=3, frameon=False)
fig.suptitle("Échecs provoqués : perte par époque (échelles différentes)", x=0.01, ha="left", fontsize=12)
fig.tight_layout()
plt.show()

Pour le réseau profond, la norme du gradient de chaque couche au premier batch, avant tout entraînement, montre où le signal se perd :

In [ ]:
def gradient_norms(hidden: list[int]) -> list[float]:
    """Norme du gradient des poids de chaque couche Linear, de l'entrée vers la sortie, sur un batch."""
    set_seed()
    model = DenseNet(N_IN, hidden, len(CLASSES))
    xb, yb = next(iter(DataLoader(datasets["train"], batch_size=BATCH_SIZE, shuffle=True)))
    loss_fn(model(xb), yb).backward()
    return [layer.weight.grad.norm().item() for layer in model.net if isinstance(layer, nn.Linear)]


deep_norms, ref_norms = gradient_norms([128] * 6), gradient_norms(HIDDEN)
grads = pd.DataFrame({"6 couches en sigmoïde": deep_norms,
                      "référence": ref_norms + [None] * (len(deep_norms) - len(ref_norms))},
                     index=[f"couche {i}" for i in range(1, len(deep_norms) + 1)])
print(f"Réseau profond : gradient de la 1re couche {deep_norms[0]:.1e}, de la sortie {deep_norms[-1]:.1e}, "
      f"soit {deep_norms[-1] / deep_norms[0]:.0f} fois plus petit à l'entrée")
grads.map(lambda v: f"{v:.1e}" if pd.notna(v) else "")

**Diagnostic des échecs provoqués**

- **Taux d'apprentissage trop fort (3)** : les pertes restent très au-dessus de ln 5 (19,6 en train et 33,5 en validation à la dernière époque) et sautent d'une époque à l'autre : chaque pas de gradient dépasse le minimum au lieu de s'en approcher. L'accuracy reste au niveau du hasard (0,273 en validation). Pas de `nan` sur ces 30 époques, mais le remède est le même : réduire le taux d'apprentissage.
- **Taux d'apprentissage trop faible (0,001)** : les deux pertes descendent lentement, sans plateau ; la perte de train vaut encore 1,346 après 30 époques, l'entraînement n'est pas fini. Mais l'écart entre train et validation reste faible (0,508 contre 0,410 en accuracy), et l'accuracy de validation dépasse celle de la référence (0,410 contre 0,347) : apprendre lentement freine le sur-apprentissage, comme un arrêt précoce. Ce résultat porte sur une seule graine et doit être confirmé sur trois ; la grille de `02_baseline` (0,01 à 0,3) n'allait pas aussi bas.
- **6 couches en sigmoïde** : la perte de train reste plate autour de ln 5 (1,627 à la dernière époque) et l'accuracy au niveau du hasard (0,200 en validation) : le réseau n'apprend rien. Les normes de gradient montrent pourquoi : avant tout entraînement, le gradient de la première couche est environ 5 800 fois plus petit que celui de la couche de sortie. La dérivée de la sigmoïde vaut au plus 1/4, et la rétropropagation la multiplie à chaque couche : le gradient s'évanouit en remontant vers l'entrée. Remèdes vus en séance 5 : ReLU et l'initialisation de He.

### 2.3 Bilan

- **Le réseau de référence sur-apprend** : la perte de validation est minimale dès l'époque 4, et l'écart entre train et validation dépasse 0,5 en accuracy à la dernière époque. Son problème est la généralisation, pas l'optimisation.
- **Chaque panne a sa signature sur les courbes** : une perte qui saute très au-dessus de ln 5 (taux trop fort), une perte qui descend lentement sans plateau (taux trop faible), une perte plate à ln 5 (réseau qui n'apprend rien, ici par évanouissement du gradient).
- **Pistes pour les corrections de la section 3** : régularisation (weight decay, dropout), arrêt précoce, et un taux d'apprentissage plus faible, qui donne 0,410 en validation sur une graine.

## 3. Corrections testées une par une

- Weight decay, dropout, arrêt précoce, batch norm, optimiseur.
- Un seul changement par run, chaque run noté dans un tableau. Un gain faible n'est retenu qu'avec la moyenne et l'écart sur 3 graines.

In [ ]:
# À compléter

## 4. Analyse des erreurs

- Œuvres mal classées et styles confondus.
- Rôle des artistes inconnus et des genres dominants.

In [ ]:
# À compléter